# Systems Data Structures

Every structure in this tier so far has been designed for **RAM**: a `dict` hashes a key and chases a pointer, a balanced tree (the **trees** notebook) keeps height ~log n so a lookup is a handful of cache-friendly comparisons. The cost model is the **comparison** (or the pointer dereference), and big-O counts them. But the structures behind **databases, filesystems, and key-value stores** live in a different world: the data is too big for RAM, so it lives on **disk**, and the unit of cost is no longer a comparison — it's a **disk seek** (milliseconds, ~100,000x a RAM access) and the **write amplification** of rewriting blocks. Under *that* cost model the in-RAM shapes are wrong, and two structures take over: the **B-tree** (read-optimized, in-place) and the **LSM tree** (write-optimized, append-only). We build small versions of both from scratch and **prove** they behave, then map out the read-vs-write trade that decides which one a system reaches for.

**Contents**

1. **What & why** — when the cost model is disk blocks, not comparisons
2. **B-tree / B+ tree** — high fan-out keeps the tree shallow (few seeks)
3. **LSM tree** — buffer writes, flush sorted runs, merge on read
4. **When to use what**
5. **Cheat-sheet** — B-tree vs LSM, read vs write amplification

## 1. What & why — when the cost model is disk blocks, not comparisons

A balanced binary search tree is the right answer in RAM: O(log n) comparisons, each one a cheap pointer hop. But put a billion keys on a disk and the assumptions collapse. The killer fact:

| Access | Rough latency | Relative |
|---|---|---|
| L1 cache | ~1 ns | 1x |
| RAM | ~100 ns | ~100x |
| SSD random read | ~100 µs | ~100,000x |
| HDD seek | ~10 ms | ~10,000,000x |

A binary tree of a billion keys is **~30 levels deep**. If each node is on a different disk page, a single lookup is **30 random seeks** — on an HDD that's a third of a *second*. The comparison count (~30) is irrelevant; the **seek count** is everything.

Two ideas reshape data structures for this world:

- **Disk I/O happens in fixed-size blocks** (a page, typically 4–16 KB) — reading 1 byte and reading 8 KB cost the same. So pack *many* keys into each node and make the tree **wide and shallow**: a high **fan-out** turns 30 levels into 3–4. That's the **B-tree**.
- **Sequential writes are vastly cheaper than random writes** (no seek between them; SSDs also suffer *write amplification* erasing/rewriting blocks for in-place updates). So never update in place — **buffer writes in RAM and flush them sequentially** as immutable sorted files, cleaning up later. That's the **LSM tree**.

The same balance problem from the **trees** notebook still lurks (an unbalanced disk tree is a disaster — every level is a seek), but the fix is different: B-trees stay balanced by **splitting full nodes**, not rotating, and they grow at the *root*, so every leaf is always the same depth.

## 2. B-tree / B+ tree — high fan-out keeps the tree shallow

A **B-tree of order `m`** is a balanced search tree where each node holds **up to `m−1` sorted keys** and **up to `m` children** between them (a node with keys `[k0, k1]` has 3 children: `< k0`, between, `> k1`). The invariants that make it tick:

- **All leaves are at the same depth** — the tree is *perfectly* height-balanced, always.
- Every node (except the root) is at least **half full** (≥ ⌈m/2⌉−1 keys), so the tree can't degenerate into a sparse vine.
- Keys within a node are sorted; a search does a small in-node scan, then descends to one child.

The magic is **fan-out**. With `m = 100`, each node has up to 100 children, so the height is `log₁₀₀(n)` — a *billion* keys fit in **~5 levels**, i.e. ~5 seeks instead of ~30. That's the entire reason B-trees, not BSTs, sit under database indexes and filesystems.

### Growing by splitting (not rotating)

A BST grows *downward* — new leaves hang off the bottom, and the path to them can get long (the balance problem). A B-tree grows **upward**: when a node fills to `m` keys, it **splits**, the middle key is pushed up into the parent, and if that overflows the root, a brand-new root is created. Because growth happens at the *root*, every leaf stays equidistant — balance is automatic, no rotations needed. We implement the classic *proactive* split: descending toward a leaf, split any full child before recursing into it.

In [1]:
import random, bisect

class BTreeNode:
    __slots__ = ("keys", "children", "leaf")
    def __init__(self, leaf=True):
        self.keys = []           # sorted keys held in this node
        self.children = []       # len == len(keys)+1 for an internal node
        self.leaf = leaf

class BTree:
    """A minimal B-tree (order m) with insert + search. Keys are unique."""
    def __init__(self, m=5):
        assert m >= 3, "order must be >= 3"
        self.m = m               # max children; max keys = m-1
        self.root = BTreeNode(leaf=True)

    def search(self, key, node=None):
        node = self.root if node is None else node
        i = bisect.bisect_left(node.keys, key)      # in-node scan
        if i < len(node.keys) and node.keys[i] == key:
            return True
        if node.leaf:
            return False
        return self.search(key, node.children[i])   # descend to one child

    def _split_child(self, parent, i):
        """Split parent.children[i] (which is full) around its median."""
        m = self.m
        full = parent.children[i]
        mid = (m - 1) // 2                           # median index
        median = full.keys[mid]
        right = BTreeNode(leaf=full.leaf)
        right.keys = full.keys[mid + 1:]             # upper half -> new right node
        full.keys = full.keys[:mid]                  # lower half stays
        if not full.leaf:
            right.children = full.children[mid + 1:]
            full.children = full.children[:mid + 1]
        parent.keys.insert(i, median)               # median rises into the parent
        parent.children.insert(i + 1, right)

    def _insert_nonfull(self, node, key):
        i = bisect.bisect_left(node.keys, key)
        if node.leaf:
            if i < len(node.keys) and node.keys[i] == key:
                return                               # ignore duplicates
            node.keys.insert(i, key)
            return
        # proactively split the child if it's full, before descending
        if len(node.children[i].keys) == self.m - 1:
            self._split_child(node, i)
            if key > node.keys[i]:
                i += 1
            elif key == node.keys[i]:
                return
        self._insert_nonfull(node.children[i], key)

    def insert(self, key):
        root = self.root
        if len(root.keys) == self.m - 1:             # root full -> tree grows upward
            new_root = BTreeNode(leaf=False)
            new_root.children.append(root)
            self._split_child(new_root, 0)
            self.root = new_root
        self._insert_nonfull(self.root, key)

    def __contains__(self, key):
        return self.search(key)

    def items(self):                                 # in-order traversal -> sorted
        out = []
        def walk(node):
            for i, k in enumerate(node.keys):
                if not node.leaf:
                    walk(node.children[i])
                out.append(k)
            if not node.leaf:
                walk(node.children[-1])
        walk(self.root)
        return out

    def height(self):
        h, node = 0, self.root
        while not node.leaf:
            h += 1
            node = node.children[0]
        return h

bt = BTree(m=5)
for k in [10, 20, 5, 6, 12, 30, 7, 17]:
    bt.insert(k)
print("12 in tree :", 12 in bt)
print("99 in tree :", 99 in bt)
print("sorted     :", bt.items())
print("height     :", bt.height(), "levels")

12 in tree : True
99 in tree : False
sorted     : [5, 6, 7, 10, 12, 17, 20, 30]
height     : 1 levels


### The fact-check: every claim against a sorted oracle

Three properties define a correct B-tree, and we assert each one against the obvious oracle — a Python `set`/`sorted` over the same random inserts:

1. **Search matches membership** — `key in btree` agrees with `key in set` for both inserted and absent keys (no false hits, no misses).
2. **In-order iteration is sorted** — walking the tree reproduces `sorted(set(...))` exactly.
3. **It stays balanced** — all leaves are at the same depth, *and* every non-root node is at least half full (the structural invariants). RNG is seeded for reproducibility.

In [2]:
random.seed(42)

def check_invariants(bt):
    """Assert all leaves same depth and every non-root node >= half full."""
    min_keys = (bt.m + 1) // 2 - 1            # ceil(m/2) - 1
    leaf_depths = []
    def walk(node, depth, is_root):
        # ordering within the node
        assert node.keys == sorted(node.keys)
        if not is_root:
            assert len(node.keys) >= min_keys, "node underflow (< half full)"
        assert len(node.keys) <= bt.m - 1, "node overflow"
        if node.leaf:
            leaf_depths.append(depth)
        else:
            assert len(node.children) == len(node.keys) + 1
            for c in node.children:
                walk(c, depth + 1, False)
    walk(bt.root, 0, True)
    assert len(set(leaf_depths)) == 1, "leaves at differing depths -> unbalanced!"
    return leaf_depths[0]

bt = BTree(m=6)
keys = random.sample(range(100_000), 4_000)       # random insert order
oracle = set()
for k in keys:
    bt.insert(k)
    oracle.add(k)

# (1) search agrees with the set oracle, on present AND absent keys
assert all(k in bt for k in keys), "missing an inserted key!"
absent = [k for k in random.sample(range(100_000), 4_000) if k not in oracle]
assert not any(k in bt for k in absent), "false positive on an absent key!"

# (2) in-order iteration == sorted oracle
assert bt.items() == sorted(oracle), "iteration not in sorted order!"

# (3) structural invariants: balanced + every node >= half full
depth = check_invariants(bt)

print(f"inserted {len(keys):,} random keys into an order-{bt.m} B-tree")
print(f"search matches set oracle      : OK ({len(absent)} absent keys also checked)")
print("in-order iteration == sorted() : OK")
print(f"all leaves at depth            : {depth}  (balanced)")
print("every non-root node half-full  : OK")
print(f"height for {len(keys):,} keys        : {bt.height()} levels"
      f"  (a binary tree would be ~{len(keys).bit_length()})")

inserted 4,000 random keys into an order-6 B-tree
search matches set oracle      : OK (3843 absent keys also checked)
in-order iteration == sorted() : OK
all leaves at depth            : 5  (balanced)
every non-root node half-full  : OK
height for 4,000 keys        : 5 levels  (a binary tree would be ~12)


### Why shallow wins: fan-out vs binary depth

The whole point is seek count. Here we hold the key set fixed and **vary the order `m`** to watch the tree get dramatically shallower as fan-out rises — and contrast against the ~log₂ n depth a balanced *binary* tree would need. On disk, each level is (worst case) one seek, so height *is* the latency.

In [3]:
import math

random.seed(0)
keys = random.sample(range(1_000_000), 50_000)

print(f"{'order m':>8}{'B-tree height':>16}{'max fan-out':>14}")
heights = {}
for m in (4, 8, 16, 64, 256):
    bt = BTree(m=m)
    for k in keys:
        bt.insert(k)
    assert bt.items() == sorted(set(keys))        # still correct at every order
    heights[m] = bt.height()
    print(f"{m:>8}{bt.height():>16}{m:>14}")

n = len(keys)
binary_depth = math.ceil(math.log2(n))
print(f"\n{n:,} keys:")
print(f"  balanced BINARY tree depth (log2 n)  : ~{binary_depth} levels -> ~{binary_depth} seeks")
print(f"  order-256 B-tree                     : {heights[256]} levels -> ~{heights[256]} seeks "
      f"({binary_depth // max(heights[256], 1)}x shallower)")
print("  reading a wide node costs ONE block I/O regardless of how many keys it holds")

 order m   B-tree height   max fan-out
       4              11             4


       8               6             8
      16               4            16
      64               2            64
     256               2           256

50,000 keys:
  balanced BINARY tree depth (log2 n)  : ~16 levels -> ~16 seeks
  order-256 B-tree                     : 2 levels -> ~2 seeks (8x shallower)
  reading a wide node costs ONE block I/O regardless of how many keys it holds


### The B+ tree variant — and why real systems use it

Production databases and filesystems use a close cousin, the **B+ tree**, with two changes that matter enormously for disk:

| | B-tree (what we built) | B+ tree (what PostgreSQL/ext4 use) |
|---|---|---|
| Where values live | in **every** node (internal + leaf) | **only in the leaves**; internal nodes hold *copy keys* as a routing index |
| Leaves | independent | **linked in a sorted chain** (each leaf points to the next) |

Both changes serve the disk cost model:

- **Values only in leaves** means internal nodes hold *just keys* — so they pack more keys per block, raising fan-out and dropping the tree even shallower. The internal levels become a pure routing index that often fits entirely in RAM/cache; only the final leaf access touches disk.
- **Linked leaves** make a **range scan** (`WHERE x BETWEEN a AND b`, or reading a directory) a single descent to the start leaf followed by a **sequential walk** along the leaf chain — no repeated root-to-leaf traversals, and sequential disk reads (the cheap kind). This is why `ORDER BY` and range queries are fast on a B+ tree index.

This is the workhorse index of the storage world: **PostgreSQL** and **MySQL/InnoDB** primary and secondary indexes, **ext4** directory htrees, **NTFS**, and **SQLite** all use B+ trees (or near variants). When a database "uses an index," it's almost always walking one of these.

In [4]:
# A tiny B+ tree leaf-chain sketch: keys routed by an index, values in linked leaves.
# We won't reimplement splitting here (the B-tree above proves that); instead we
# demonstrate the property that MATTERS -- a range scan via the linked leaf chain.

class BPlusLeaf:
    __slots__ = ("keys", "values", "next")
    def __init__(self):
        self.keys, self.values, self.next = [], [], None

# Build sorted leaves of fan-out 4, chained together (as a B+ tree maintains them).
random.seed(3)
pairs = sorted((k, f"v{k}") for k in random.sample(range(1000), 40))
leaves, FAN = [], 4
for i in range(0, len(pairs), FAN):
    leaf = BPlusLeaf()
    for k, v in pairs[i:i + FAN]:
        leaf.keys.append(k); leaf.values.append(v)
    leaves.append(leaf)
for a, b in zip(leaves, leaves[1:]):
    a.next = b                                       # link the chain

def range_scan(leaves, lo, hi):
    """Descend to the start leaf (binary search the leaf list), then walk the chain."""
    start = bisect.bisect_right([lf.keys[0] for lf in leaves], lo) - 1
    start = max(start, 0)
    out, leaf = [], leaves[start]
    while leaf is not None:                           # SEQUENTIAL walk -- the cheap I/O
        for k, v in zip(leaf.keys, leaf.values):
            if lo <= k <= hi:
                out.append((k, v))
        if leaf.keys[-1] > hi:
            break
        leaf = leaf.next
    return out

result = range_scan(leaves, 300, 500)
oracle = [(k, v) for k, v in pairs if 300 <= k <= 500]
assert result == oracle, "range scan disagreed with the sorted oracle!"
print(f"range scan [300, 500] via leaf chain: {len(result)} keys")
print("  ", [k for k, _ in result])
print("matches sorted oracle              :", result == oracle)
print("(one descent + a sequential chain walk -- not one root-to-leaf trip per key)")

range scan [300, 500] via leaf chain: 7 keys
   [378, 399, 406, 480, 481, 485, 487]
matches sorted oracle              : True
(one descent + a sequential chain walk -- not one root-to-leaf trip per key)


## 3. LSM tree — buffer writes, flush sorted runs, merge on read

The B-tree updates **in place**: changing a key means seeking to its block and rewriting it. That's fine for reads, but every write is a random write — exactly the expensive operation on disk and the one that wears out SSDs (write amplification). For **write-heavy** workloads — event logs, time series, metrics, messaging — a different shape wins: the **log-structured merge tree (LSM tree)**, which never updates in place and turns random writes into sequential ones.

The architecture is three moving parts:

- **Memtable** — an in-RAM sorted structure (a balanced tree / skip list — see the **probabilistic structures** notebook's skip list) absorbing all writes. Fast, no disk.
- **SSTables (sorted runs)** — when the memtable fills, it's **flushed to disk as one immutable, sorted file** in a single sequential write. Never modified again.
- **Compaction** — a background process **merges** runs together, discarding overwritten/deleted keys, to stop reads from having to consult too many files.

The defining tricks:

- **Updates and deletes are just writes.** Overwriting a key appends a newer version; deleting writes a **tombstone** marker. Nothing is edited in place.
- **Newest wins.** A read checks the memtable first, then runs **newest-to-oldest**, and returns the **first** version it finds — that's by definition the latest write.
- **Tombstones suppress.** If the newest version found is a tombstone, the key is reported absent (even though older runs still physically contain it, until compaction garbage-collects them).

In [5]:
_TOMBSTONE = object()    # sentinel marking a deleted key

class LSMTree:
    """In-memory simulation of an LSM tree: memtable + immutable sorted runs."""
    def __init__(self, memtable_limit=64):
        self.memtable = {}                 # active in-RAM buffer (key -> value/tombstone)
        self.runs = []                     # list of immutable runs, OLDEST first
        self.limit = memtable_limit

    def put(self, key, value):
        self.memtable[key] = value
        if len(self.memtable) >= self.limit:
            self._flush()

    def delete(self, key):
        self.memtable[key] = _TOMBSTONE   # a delete is just a tombstone write
        if len(self.memtable) >= self.limit:
            self._flush()

    def _flush(self):
        # Flush the memtable to an immutable, sorted run (one sequential write).
        run = dict(sorted(self.memtable.items()))
        self.runs.append(run)
        self.memtable = {}

    def get(self, key):
        # memtable first, then runs NEWEST -> OLDEST; first hit wins.
        if key in self.memtable:
            v = self.memtable[key]
            return None if v is _TOMBSTONE else v
        for run in reversed(self.runs):           # newest run last in the list
            if key in run:
                v = run[key]
                return None if v is _TOMBSTONE else v
        return None                                # never seen

    def compact(self):
        """Merge all runs newest-first into one; drop shadowed keys & tombstones."""
        merged = {}
        # apply memtable + runs newest -> oldest, keeping only the first (newest) seen
        sources = [self.memtable] + list(reversed(self.runs))
        for src in sources:
            for k, v in src.items():
                if k not in merged:
                    merged[k] = v
        # a fully-merged tree can physically drop tombstones (no older version remains)
        cleaned = {k: v for k, v in merged.items() if v is not _TOMBSTONE}
        self.runs = [dict(sorted(cleaned.items()))] if cleaned else []
        self.memtable = {}

    def num_runs(self):
        return len(self.runs)

lsm = LSMTree(memtable_limit=4)
lsm.put("a", 1); lsm.put("b", 2); lsm.put("a", 10)   # overwrite a
lsm.delete("b")                                       # tombstone b
print("get a :", lsm.get("a"), " (newest write wins)")
print("get b :", lsm.get("b"), " (tombstoned -> absent)")
print("get z :", lsm.get("z"), " (never written)")
print("runs  :", lsm.num_runs())

get a : 10  (newest write wins)
get b : None  (tombstoned -> absent)
get z : None  (never written)
runs  : 0


### The fact-check: reads always return the latest write

This is the one claim an LSM tree must never get wrong: across an arbitrary interleaving of `put` / `delete` / `get` — with overwrites, deletes, re-inserts, flushes happening mid-stream, and keys scattered across the memtable and many runs — a `get` must **always** return the result of the **most recent** `put`/`delete` for that key. Our oracle is a plain `dict` updated alongside every mutation; we assert agreement after *every* operation, not just at the end.

In [6]:
random.seed(7)

lsm = LSMTree(memtable_limit=8)
oracle = {}                                   # the ground truth: a plain dict
keys = [f"k{i}" for i in range(40)]
ops = 0

for step in range(5_000):
    key = random.choice(keys)
    roll = random.random()
    if roll < 0.6:                            # put
        val = random.randint(0, 10_000)
        lsm.put(key, val)
        oracle[key] = val
    elif roll < 0.8:                          # delete
        lsm.delete(key)
        oracle.pop(key, None)
    else:                                     # get -> must match oracle EXACTLY
        ops += 1
        assert lsm.get(key) == oracle.get(key), f"stale read for {key}!"
    # occasionally trigger a compaction mid-stream; reads must still be correct
    if step % 1500 == 1499:
        lsm.compact()

# final full sweep over EVERY key, across all runs + memtable
for key in keys:
    assert lsm.get(key) == oracle.get(key), f"final mismatch for {key}"

print(f"ran 5,000 mixed put/delete/get ops ({ops} reads verified inline)")
print("every read returned the latest write : OK")
print(f"final state matches dict oracle      : OK ({len(oracle)} live keys)")
print(f"runs on disk after compaction        : {lsm.num_runs()}")

ran 5,000 mixed put/delete/get ops (1004 reads verified inline)
every read returned the latest write : OK
final state matches dict oracle      : OK (32 live keys)
runs on disk after compaction        : 46


### Read amplification, Bloom filters, and compaction

The LSM's write advantage has a cost on the **read** side. A key might live in any run, so a `get` for an *absent* key would, naively, have to check **every** run — that's **read amplification**, and it grows with the number of runs. Two mechanisms fight it:

- **Compaction** merges many small runs into fewer big ones, so reads consult fewer files. The demo below shows runs piling up under a write storm, then collapsing after a compaction.
- **Per-run Bloom filters.** Each run keeps a **Bloom filter** of its keys (see the **probabilistic structures** notebook — membership with no false negatives). Before touching a run's data, a `get` asks its Bloom filter "could this key be here?"; a "no" is *certain*, so the run is **skipped** entirely. Since a Bloom filter never gives a false negative, this can never hide a real key — it only ever saves work. We measure how many run-probes the Bloom filters skip on absent-key lookups.

In [7]:
import hashlib

class BloomFilter:                            # compact: see probabilistic-structures
    def __init__(self, m_bits, k):
        self.m, self.k = m_bits, k
        self.bits = bytearray((m_bits + 7) // 8)
    def _idx(self, item):
        d = hashlib.sha256(str(item).encode()).digest()
        h1 = int.from_bytes(d[:8], "big"); h2 = int.from_bytes(d[8:16], "big")
        for i in range(self.k):
            yield (h1 + i * h2) % self.m
    def add(self, item):
        for p in self._idx(item):
            self.bits[p >> 3] |= 1 << (p & 7)
    def __contains__(self, item):
        return all(self.bits[p >> 3] & (1 << (p & 7)) for p in self._idx(item))

# Write a storm of distinct keys into small memtables -> many runs accumulate.
random.seed(11)
lsm = LSMTree(memtable_limit=50)
written = random.sample(range(200_000), 2_000)
for k in written:
    lsm.put(k, k * 2)
print(f"after writing {len(written):,} keys: {lsm.num_runs()} runs accumulated")

# Build one Bloom filter per run (no false negatives -> a 'no' lets us SKIP the run).
filters = []
for run in lsm.runs:
    bf = BloomFilter(m_bits=8 * len(run), k=5)
    for key in run:
        bf.add(key)
    filters.append(bf)

# Look up 5,000 ABSENT keys; count run-probes saved by Bloom 'no' answers.
absent = [k for k in random.sample(range(200_000, 400_000), 5_000)]
probes_without = probes_with = false_neg = 0
for key in absent:
    for run, bf in zip(lsm.runs, filters):
        probes_without += 1                  # naive: must check every run
        if key in bf:                        # Bloom says "maybe" -> actually check
            probes_with += 1
            if key in run:                   # would be a Bloom false-negative bug
                false_neg += 1

assert false_neg == 0, "Bloom filter hid a real key -- impossible!"
assert all(lsm.get(k) is None for k in absent)        # absent keys really are absent
saved = (1 - probes_with / probes_without) * 100
print(f"absent-key lookups: {len(absent):,}")
print(f"  run-probes without Bloom : {probes_without:,}")
print(f"  run-probes WITH Bloom    : {probes_with:,}  (skipped {saved:.1f}% of run reads)")
print(f"  Bloom false negatives    : {false_neg}  (guaranteed 0)")

# Compaction collapses the runs -> read amplification drops at the source.
before = lsm.num_runs()
lsm.compact()
print(f"compaction: {before} runs -> {lsm.num_runs()} run")
assert all(lsm.get(k) == k * 2 for k in written), "compaction lost a key!"
print("all live keys still correct after compaction : OK")

after writing 2,000 keys: 40 runs accumulated


absent-key lookups: 5,000
  run-probes without Bloom : 200,000
  run-probes WITH Bloom    : 4,913  (skipped 97.5% of run reads)
  Bloom false negatives    : 0  (guaranteed 0)
compaction: 40 runs -> 1 run
all live keys still correct after compaction : OK


### B-tree vs LSM tree — the core trade

The two structures sit at opposite ends of the **read-vs-write amplification** spectrum, and that single axis decides which a system picks:

| | **B-tree** | **LSM tree** |
|---|---|---|
| Write path | find block, **update in place** (random write) | append to memtable, flush **sequentially** |
| Write amplification | high (rewrite a full block per change) | low (sequential, batched) |
| Read path | one root-to-leaf descent (few seeks) | check memtable + several runs (more seeks) |
| Read amplification | low | higher (mitigated by Bloom filters + compaction) |
| Space | compact, in place | extra copies until compaction reclaims |
| Best for | **read-heavy**, point + range queries | **write-heavy**, ingest-heavy streams |
| Used by | PostgreSQL, MySQL/InnoDB, ext4, SQLite | **RocksDB, LevelDB, Cassandra**, ScyllaDB, HBase |

The names tell the story: a relational database's default index is a B+ tree (read-optimized, transactional point/range queries); the storage engines behind high-ingest key-value and wide-column stores — RocksDB, LevelDB, Cassandra — are LSM trees (write-optimized, append-only). Many modern systems make it **pluggable**: MySQL can run InnoDB (B-tree) or MyRocks (LSM) under the same SQL, chosen by whether the workload reads or writes more.

## 4. When to use what

| You need... | Reach for | Instead of | Why |
|---|---|---|---|
| An on-disk ordered index, read-heavy | **B+ tree** | in-RAM BST / AVL | high fan-out = few seeks; in-place point + range queries |
| Fast range scans / `ORDER BY` on disk | **B+ tree** (linked leaves) | hash index | one descent + sequential leaf-chain walk |
| A write-heavy / ingest-heavy store | **LSM tree** | B-tree | sequential writes, low write amplification |
| To skip runs on absent-key reads | per-run **Bloom filter** | scanning every run | "no" is certain; never hides a real key |
| To cap LSM read amplification | **compaction** | unbounded run count | merges runs, drops tombstones |
| An in-RAM ordered map (fits in memory) | balanced tree / `sortedcontainers` | B-tree / LSM | disk structures add overhead with no payoff in RAM |
| In-RAM membership / lookup | `dict` / `set` | any tree | O(1) average — the **trees** notebook's bottom line |
| Approximate membership at scale | **Bloom filter** alone | a full index | bits per key; see **probabilistic structures** |

## 5. Cheat-sheet — B-tree vs LSM, read vs write amplification

<sub>n = total keys, m = B-tree order (fan-out), B = disk block size, R = number of LSM runs. "I/O" = disk block reads/writes, the cost that actually matters.</sub>

| Operation | **B-tree / B+ tree** | **LSM tree** |
|---|:---:|:---:|
| Point lookup | O(log_m n) I/O | O(R) runs, ~O(1) with Bloom filters |
| Range scan | O(log_m n + k/B) (linked leaves) | O(R · scan) — merge across runs |
| Insert / update | O(log_m n) I/O, **in place** | O(1) amortized (append) |
| Delete | O(log_m n) I/O, in place | O(1) (write a tombstone) |
| Write amplification | **high** (rewrite a block per write) | **low** (sequential, batched) |
| Read amplification | **low** (one descent) | **higher** (check R runs) |
| Space amplification | low (compact) | higher (dup versions until compaction) |
| Height for 1e9 keys, m=256 | ~4 levels (~4 seeks) | n/a (runs, not depth) |

**The one-line mental model:** in RAM you count comparisons and reach for a hash map or a balanced tree (the **trees** notebook); on disk you count **seeks and rewrites**, and the question becomes *do I read more or write more?* Read-heavy ⇒ **B-tree** (wide and shallow, update in place, the index under every SQL database). Write-heavy ⇒ **LSM tree** (buffer in RAM, flush sequentially, merge lazily, skip runs with **Bloom filters**). Same data, opposite optimizations — pick by which amplification you can least afford.